# EDA: flats_buildings_raw → flats_buildings_clean

Analysis of the joined dataset produced by the ETL DAG
(`prepare_flats_buildings_dataset` → `public.flats_buildings_raw`).
Decides which cleaning steps to apply. Cleaning functions live in
`part1_airlfow/plugins/cleaning.py` and are reused by the cleaning DAG
`clean_flats_buildings_dataset`.

**Source:** Postgres из `.env` (DB_DESTINATION_*).
**Snapshot:** 141362 rows, joined from buildings (24620) + flats (141362).

In [ ]:
import sys, pathlib
import os, pandas as pd
from sqlalchemy import create_engine

# Import shared cleaning functions (used by DAG too)
REPO_ROOT = pathlib.Path.cwd().resolve()
while not (REPO_ROOT / '.gitignore').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'part1_airlfow' / 'plugins'))
from cleaning import (
    remove_duplicate_flats, drop_logical_impossibilities,
    clip_ceiling_height, drop_price_outliers, clean_flats_buildings,
)

# Load .env
cfg = {}
for line in open(REPO_ROOT / '.env'):
    s = line.strip()
    if not s or s.startswith('#') or '=' not in s: continue
    k, v = s.split('=', 1); cfg[k.strip()] = v.strip()
dsn = (f"postgresql://{cfg['DB_DESTINATION_USER']}:{cfg['DB_DESTINATION_PASSWORD']}"
       f"@{cfg['DB_DESTINATION_HOST']}:{cfg['DB_DESTINATION_PORT']}/{cfg['DB_DESTINATION_NAME']}")
eng = create_engine(dsn)

df = pd.read_sql('SELECT * FROM public.flats_buildings_raw', eng)
print(f'rows: {len(df)}, cols: {len(df.columns)}')
df.head()

## 1. Duplicates & nulls

In [ ]:
print('dup flat_id:', len(df) - df['flat_id'].nunique())
print('\nnulls per col:')
print(df.isna().sum().sort_values(ascending=False).head(10))

## 2. Numeric ranges (decide caps)

In [ ]:
df.describe(percentiles=[0.01, 0.5, 0.99, 0.995]).T[['min','1%','50%','99%','99.5%','max']]

## 3. Logical impossibilities

In [ ]:
issues = {
    'kitchen_area > total_area': (df['kitchen_area'] > df['total_area']).sum(),
    'living_area > total_area':  (df['living_area']  > df['total_area']).sum(),
    'floor > floors_total':      (df['floor']        > df['floors_total']).sum(),
    'price <= 0':                (df['price']        <= 0).sum(),
    'total_area <= 0':           (df['total_area']   <= 0).sum(),
    'ceiling_height > 5':        (df['ceiling_height'] > 5).sum(),
    'price > 1_000_000_000':     (df['price'] > 1_000_000_000).sum(),
}
pd.Series(issues, name='count')

## 4. Apply cleaning pipeline

Steps (defined in `plugins/cleaning.py`):
1. `remove_duplicate_flats` — dedupe by flat_id (no-op for current snapshot)
2. `drop_logical_impossibilities` — kitchen/living > total → drop
3. `drop_price_outliers` — drop price above p99.5 (placeholder values)
4. `clip_ceiling_height` — clip to [2.0, 5.0] m

In [ ]:
clean = clean_flats_buildings(df)
print(f'raw:   {len(df)} rows')
print(f'clean: {len(clean)} rows  (dropped {len(df) - len(clean)})')
print('\nceiling_height after clip:')
print(clean['ceiling_height'].describe())
print('\nprice after drop:')
print(clean['price'].describe())

## 5. Sanity check on clean output

In [ ]:
post_issues = {
    'kitchen_area > total_area': (clean['kitchen_area'] > clean['total_area']).sum(),
    'ceiling_height < 2.0':      (clean['ceiling_height'] < 2.0).sum(),
    'ceiling_height > 5.0':      (clean['ceiling_height'] > 5.0).sum(),
    'price > 1_000_000_000':     (clean['price'] > 1_000_000_000).sum(),
}
pd.Series(post_issues, name='count_after_clean')